<a href="https://colab.research.google.com/github/soumyasinghcs-debug/AI-298/blob/main/week9_forward_chaining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:

from itertools import count

KB = {
    ("Student", "Ravi"),
    ("Studies", "Ravi")
}


rules = [
    (
        [("Student", "?x"), ("Studies", "?x")],
        ("Attends", "?x")
    )
]


def is_variable(x):
    return isinstance(x, str) and x.startswith("?")


def subst(theta, term):
    if isinstance(term, tuple):
        return tuple(subst(theta, x) for x in term)

    while is_variable(term) and term in theta:
        term = theta[term]

    return term


def unify(x, y, theta=None):
    if theta is None:
        theta = {}

    x = subst(theta, x)
    y = subst(theta, y)

    if x == y:
        return theta

    if is_variable(x):
        theta = theta.copy()
        theta[x] = y
        return theta

    if is_variable(y):
        theta = theta.copy()
        theta[y] = x
        return theta

    if isinstance(x, tuple) and isinstance(y, tuple):
        if len(x) != len(y):
            return None

        for a, b in zip(x, y):
            theta = unify(a, b, theta)
            if theta is None:
                return None

        return theta

    return None


counter = count()

def standardize(rule):
    premises, conclusion = rule
    n = next(counter)

    variables = {}

    def rename(term):
        if isinstance(term, tuple):
            return tuple(rename(x) for x in term)

        if is_variable(term):
            if term not in variables:
                variables[term] = term + "_" + str(n)
            return variables[term]

        return term

    return (
        [rename(p) for p in premises],
        rename(conclusion)
    )


def find_substitutions(premises, facts, theta=None):
    if theta is None:
        theta = {}

    if not premises:
        return [theta]

    first = premises[0]
    remaining = premises[1:]

    results = []

    for fact in facts:
        new_theta = unify(first, fact, theta)

        if new_theta is not None:
            results.extend(
                find_substitutions(
                    remaining, facts, new_theta
                )
            )

    return results


def FOL_FC_ASK(KB, rules, alpha):

    new = set()


    for fact in KB:
        answer = unify(alpha, fact)
        if answer is not None:
            return answer

    while True:
        new = set()

        for rule in rules:


            premises, conclusion = standardize(rule)


            substitutions = find_substitutions(
                premises, KB
            )

            for theta in substitutions:


                q = subst(theta, conclusion)


                if q not in KB and q not in new:

                    new.add(q)


                    phi = unify(alpha, q)

                    if phi is not None:
                        KB.update(new)
                        return phi


        if not new:
            break

        KB.update(new)

    return False


alpha = ("Attends", "Ravi")


result = FOL_FC_ASK(KB, rules, alpha)

print("Knowledge Base:", KB)

if result is not False:
    print("Query:", alpha)
    print("Result: TRUE")
    print("Substitution:", result)
else:
    print("Result: FALSE")


Knowledge Base: {('Attends', 'Ravi'), ('Student', 'Ravi'), ('Studies', 'Ravi')}
Query: ('Attends', 'Ravi')
Result: TRUE
Substitution: {}
